### Transform Orders Data - Explode Arrays

 1. Access elements from JSON Object
 2. Deduplicate Array Elements
 3. Explode Arrays
 4. Write the Transformed Data to Silver Schema

### 1. Access elements from JSON Object

In [0]:
select * from gizmobox_sivan.silver.orders_json

In [0]:
select
    json_value.order_id,
    json_value.order_status,
    json_value.payment_method,
    json_value.total_amount,
    json_value.transaction_timestamp,
    json_value.customer_id,
    json_value.items
from gizmobox_sivan.silver.orders_json

### 2. Deduplicate Array Elements

[function - array_distinct](https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/functions/array_distinct)

In [0]:
select
    json_value.order_id,
    json_value.order_status,
    json_value.payment_method,
    json_value.total_amount,
    json_value.transaction_timestamp,
    json_value.customer_id,
    array_distinct(json_value.items) as items -- Ex: In prev statement, 10th row was having duplicate items, 2nd row was having 2 distinct items
from gizmobox_sivan.silver.orders_json

### 3. Explore Arrays

Function [explode](https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/functions/explode)

In [0]:
select
    json_value.order_id,
    json_value.order_status,
    json_value.payment_method,
    json_value.total_amount,
    json_value.transaction_timestamp,
    json_value.customer_id,
    explode(array_distinct(json_value.items)) as item
from gizmobox_sivan.silver.orders_json

In [0]:
CREATE OR REPLACE TEMP VIEW tv_orders_exploded AS
SELECT
    json_value.order_id,
    json_value.order_status,
    json_value.payment_method,
    json_value.total_amount,
    json_value.transaction_timestamp,
    json_value.customer_id,
    explode(array_distinct(json_value.items)) as item
from gizmobox_sivan.silver.orders_json

In [0]:
select * from tv_orders_exploded

In [0]:
select 
    order_id,
    order_status,
    payment_method,
    total_amount,
    transaction_timestamp,
    customer_id,
    item.category,    
    item.details,
    item.details.brand,
    item.details.color,
    item.item_id,
    item.name,
    item.price,
    item.quantity
from tv_orders_exploded

In [0]:
CREATE TABLE IF NOT EXISTS gizmobox_sivan.silver.orders
select 
    order_id,
    order_status,
    payment_method,
    total_amount,
    transaction_timestamp,
    customer_id,
    item.category,        
    item.details.brand,
    item.details.color,
    item.item_id,
    item.name,
    item.price,
    item.quantity
from tv_orders_exploded

In [0]:
select * from gizmobox_sivan.silver.orders